# Random Forest baseline

Two feature variants per level:
- **deep:** mean-pooled multilevel deep features (one vector per crop)
- **raw:** flattened RGB+Gabor(+Sobel) features → reduced via PCA to 256 dims (raw is otherwise ~32k-dim per crop, infeasible for RF)

Tail-merging + class-weighting are applied for L2/L3. L1 uses class-balanced RF without tail-merge.

In [2]:
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

from fgvd_utils import ExperimentConfig, run_experiment

CKPT_ROOT = Path('checkpoints')
PLOT_ROOT = Path('plots')

## L1 — deep features

In [2]:
cfg = ExperimentConfig(method='rf', level='L1', feature_source='deep',
                       rf_n_estimators=200, rf_max_depth=20)
out = run_experiment(cfg, ckpt_root=CKPT_ROOT, plot_root=PLOT_ROOT)
out['metrics']


=== RF | L1 | deep | hierarchical=False | tail_min=None ===
Samples: train=15702 val=3850 test=4890 | classes=7
Building features (RF/deep) ...
  X_tr=(15702, 64)  X_va=(3850, 64)  X_te=(4890, 64)
              precision    recall  f1-score   support

autorickshaw     0.9113    0.8992    0.9052       754
         bus     0.8156    0.6667    0.7337       219
         car     0.8579    0.9487    0.9010      1559
    mini-bus     0.7600    0.3393    0.4691        56
  motorcycle     0.8000    0.8708    0.8339      1084
     scooter     0.8490    0.7239    0.7815       909
       truck     0.7757    0.6602    0.7133       309

    accuracy                         0.8442      4890
   macro avg     0.8242    0.7298    0.7625      4890
weighted avg     0.8434    0.8442    0.8403      4890



{'val_acc': 0.8431168831168832,
 'acc': 0.8441717791411043,
 'top1_acc': 0.8441717791411043,
 'top3_acc': 0.9791411042944785,
 'top5_acc': 0.9973415132924336,
 'macro_f1': 0.7625282748306859,
 'weighted_f1': 0.8402605470291921}

## L1 — raw features (PCA→256)

In [4]:
cfg = ExperimentConfig(method='rf', level='L1', feature_source='raw',
                       rf_n_estimators=200, rf_max_depth=20, rf_pca_dim=256)
out = run_experiment(cfg, ckpt_root=CKPT_ROOT, plot_root=PLOT_ROOT)
out['metrics']


=== RF | L1 | raw | hierarchical=False | tail_min=None ===
Samples: train=15702 val=3857 test=4891 | classes=7
Building features (RF/raw) ...
  X_tr=(15702, 32768)  X_va=(3857, 32768)  X_te=(4891, 32768)
  PCA -> 256 dims (explained var=0.8031)
              precision    recall  f1-score   support

autorickshaw     0.8085    0.6777    0.7374       754
         bus     0.8413    0.2420    0.3759       219
         car     0.5983    0.8743    0.7105      1559
    mini-bus     1.0000    0.0536    0.1017        56
  motorcycle     0.5610    0.8092    0.6626      1085
     scooter     0.6894    0.2222    0.3361       909
       truck     0.8947    0.1650    0.2787       309

    accuracy                         0.6258      4891
   macro avg     0.7705    0.4349    0.4575      4891
weighted avg     0.6736    0.6258    0.5852      4891



{'val_acc': 0.6354679802955665,
 'acc': 0.6258433858106727,
 'top1_acc': 0.6258433858106727,
 'top3_acc': 0.9014516458801881,
 'top5_acc': 0.978531997546514,
 'macro_f1': 0.4575489391053448,
 'weighted_f1': 0.5851949220612934}

## L2 — deep features

In [4]:
cfg = ExperimentConfig(method='rf', level='L2', feature_source='deep',
                       rf_n_estimators=300, rf_max_depth=30)
out = run_experiment(cfg, ckpt_root=CKPT_ROOT, plot_root=PLOT_ROOT)
out['metrics']


=== RF | L2 | deep | hierarchical=True | tail_min=20 ===
Samples: train=15702 val=3850 test=4890 | classes=46
Tail-merge: kept 41/59 classes | coverage train=0.991 val=0.990 test=0.992
Building features (RF/deep) ...
  X_tr=(15702, 64)  X_va=(3850, 64)  X_te=(4890, 64)
                          precision    recall  f1-score   support

     autorickshaw::Bajaj     0.6419    0.4948    0.5588       192
    autorickshaw::Others     0.6511    0.8004    0.7181       471
   autorickshaw::Piaggio     0.8158    0.4921    0.6139        63
       autorickshaw::TVS     0.7500    0.1429    0.2400        21
     autorickshaw::other     1.0000    0.5714    0.7273         7
                bus::bus     0.6064    0.7808    0.6826       219
          car::Chevrolet     1.0000    0.2308    0.3750        13
            car::Covered     0.8889    0.5333    0.6667        15
               car::Ford     0.4500    0.3273    0.3789        55
              car::Honda     0.6875    0.2500    0.3667        44
  

{'val_acc': 0.5454545454545454,
 'acc': 0.5222903885480572,
 'top1_acc': 0.5222903885480572,
 'top3_acc': 0.7533742331288343,
 'top5_acc': 0.8462167689161554,
 'macro_f1': 0.3255153815422071,
 'weighted_f1': 0.4787947217101987}

## L2 — raw features (PCA→256)

In [5]:
cfg = ExperimentConfig(method='rf', level='L2', feature_source='raw',
                       rf_n_estimators=300, rf_max_depth=30, rf_pca_dim=256)
out = run_experiment(cfg, ckpt_root=CKPT_ROOT, plot_root=PLOT_ROOT)
out['metrics']


=== RF | L2 | raw | hierarchical=True | tail_min=20 ===
Samples: train=15702 val=3857 test=4891 | classes=46
Tail-merge: kept 41/59 classes | coverage train=0.991 val=0.990 test=0.992
Building features (RF/raw) ...
  X_tr=(15702, 28672)  X_va=(3857, 28672)  X_te=(4891, 28672)
  PCA -> 256 dims (explained var=0.8543)
                          precision    recall  f1-score   support

     autorickshaw::Bajaj     0.5234    0.2917    0.3746       192
    autorickshaw::Others     0.4112    0.6837    0.5136       471
   autorickshaw::Piaggio     0.7143    0.1587    0.2597        63
       autorickshaw::TVS     1.0000    0.1429    0.2500        21
     autorickshaw::other     1.0000    0.4286    0.6000         7
                bus::bus     0.3291    0.5890    0.4223       219
          car::Chevrolet     1.0000    0.2308    0.3750        13
            car::Covered     0.6000    0.2000    0.3000        15
               car::Ford     0.4348    0.1818    0.2564        55
              car::H

{'val_acc': 0.3533834586466165,
 'acc': 0.33408300960948684,
 'top1_acc': 0.33408300960948684,
 'top3_acc': 0.5553056634635044,
 'top5_acc': 0.6751175628705787,
 'macro_f1': 0.22081330760823656,
 'weighted_f1': 0.3024755314703992}

## L3 — deep features

In [3]:
cfg = ExperimentConfig(method='rf', level='L3', feature_source='deep',
                       rf_n_estimators=400, rf_max_depth=40)
out = run_experiment(cfg, ckpt_root=CKPT_ROOT, plot_root=PLOT_ROOT)
out['metrics']


=== RF | L3 | deep | hierarchical=True | tail_min=20 ===
Samples: train=15702 val=3850 test=4890 | classes=135
Tail-merge: kept 93/209 classes | coverage train=0.955 val=0.951 test=0.953
Building features (RF/deep) ...
  X_tr=(15702, 64)  X_va=(3850, 64)  X_te=(4890, 64)
                                      precision    recall  f1-score   support

           autorickshaw::Atul::other     0.0000    0.0000    0.0000         2
          autorickshaw::Bajaj::Bajaj     0.6351    0.4896    0.5529       192
        autorickshaw::Covered::other     0.6667    1.0000    0.8000         2
       autorickshaw::Mahindra::other     0.6667    0.6667    0.6667         3
        autorickshaw::Others::Others     0.6388    0.8110    0.7147       471
      autorickshaw::Piaggio::Piaggio     0.8095    0.5397    0.6476        63
              autorickshaw::TVS::TVS     0.6667    0.1905    0.2963        21
                       bus::bus::bus     0.6175    0.8037    0.6984       219
                    car:

{'val_acc': 0.4787012987012987,
 'acc': 0.4572597137014315,
 'top1_acc': 0.4572597137014315,
 'top3_acc': 0.6259713701431493,
 'top5_acc': 0.7034764826175869,
 'macro_f1': 0.2699362936206548,
 'weighted_f1': 0.4247843577811917}

## L3 — raw features (PCA→256)

In [6]:
cfg = ExperimentConfig(method='rf', level='L3', feature_source='raw',
                       rf_n_estimators=400, rf_max_depth=40, rf_pca_dim=256)
out = run_experiment(cfg, ckpt_root=CKPT_ROOT, plot_root=PLOT_ROOT)
out['metrics']


=== RF | L3 | raw | hierarchical=True | tail_min=20 ===
Samples: train=15702 val=3857 test=4891 | classes=135
Tail-merge: kept 93/209 classes | coverage train=0.955 val=0.949 test=0.953
Building features (RF/raw) ...
  X_tr=(15702, 28672)  X_va=(3857, 28672)  X_te=(4891, 28672)
  PCA -> 256 dims (explained var=0.8543)
                                      precision    recall  f1-score   support

           autorickshaw::Atul::other     0.0000    0.0000    0.0000         2
          autorickshaw::Bajaj::Bajaj     0.2841    0.2604    0.2717       192
        autorickshaw::Covered::other     0.6667    1.0000    0.8000         2
       autorickshaw::Mahindra::other     1.0000    0.3333    0.5000         3
        autorickshaw::Others::Others     0.4512    0.5499    0.4957       471
      autorickshaw::Piaggio::Piaggio     0.1923    0.0794    0.1124        63
              autorickshaw::TVS::TVS     0.3750    0.2857    0.3243        21
                       bus::bus::bus     0.3149    0.4

{'val_acc': 0.22841586725434276,
 'acc': 0.21958699652422817,
 'top1_acc': 0.21958699652422817,
 'top3_acc': 0.338989981598855,
 'top5_acc': 0.4113678184420364,
 'macro_f1': 0.15087571622693186,
 'weighted_f1': 0.20977613806782935}